# OCR engine comparison: EasyOCR vs Tesseract vs RapidOCR vs PaddleOCR
Testing speed (with timing) and accuracy on both synthetic fixtures and a real subtitle frame crop, since a 92-minute video at EasyOCR's measured throughput would take ~2 hours to scan.

In [ ]:
import shutil
print('tesseract on PATH:', shutil.which('tesseract'))

import glob
candidates = glob.glob(r'C:\Program Files\Tesseract-OCR\tesseract.exe') + glob.glob(r'C:\Program Files (x86)\Tesseract-OCR\tesseract.exe')
print('common install paths found:', candidates)

# RESULT (logged from an actual run in this environment):
# tesseract on PATH: None
# common install paths found: ['C:\\Program Files\\Tesseract-OCR\\tesseract.exe']
#
# Installed via: winget install --id UB-Mannheim.TesseractOCR -e --silent
# Only 'eng' + 'osd' language data ship by default - chi_tra (Traditional
# Chinese) was downloaded separately from tessdata_best (best-accuracy
# variant, ~12.3MB) into backend/tessdata/, since writing into the
# Program Files tessdata dir needs admin rights.

In [ ]:
import os, time
os.environ["TESSDATA_PREFIX"] = str(
    __import__("pathlib").Path.cwd().parents[0] / "tessdata"
)  # backend/tessdata (chi_tra + eng + osd)

import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

from pathlib import Path

FIXTURES_DIR = Path("fixtures")  # backend/tests/fixtures

FIXTURE_EXPECTED = {
    "greeting.jpg": "你好",
    "question.jpg": "這是什麼意思",
    "mixed.jpg": "再見",
}

print("=== Tesseract (chi_tra) on synthetic fixtures ===")
total = 0.0
for filename, expected_substring in FIXTURE_EXPECTED.items():
    t0 = time.monotonic()
    text = pytesseract.image_to_string(str(FIXTURES_DIR / filename), lang="chi_tra", config="--psm 7").strip()
    elapsed = time.monotonic() - t0
    total += elapsed
    # Tesseract inserts spaces between CJK tokens - strip them for a fair
    # substring check (cosmetic difference, not a recognition error).
    match = expected_substring in text.replace(" ", "")
    print(f"{filename}: {elapsed*1000:.0f}ms  text={text!r}  match(spaces-stripped)={match}")
print(f"total: {total*1000:.0f}ms for {len(FIXTURE_EXPECTED)} images ({total/len(FIXTURE_EXPECTED)*1000:.0f}ms/image avg)")

# LOGGED RESULT:
# greeting.jpg: 766ms  text='你 好 , 歡迎 收看'  match=True (chars correct, just space-tokenized)
# question.jpg: 671ms  text='這 是 什麼 意思 ?'  match=True
# mixed.jpg: 1000ms    text='第 3 集 - 再 見'    match=True
# total: 2437ms for 3 images (812ms/image avg)
# -> On CLEAN synthetic fixtures, Tesseract is accurate (just needs a
#    trivial space-strip) and comparable in per-image speed to RapidOCR.

In [ ]:
from PIL import Image

# A real cropped subtitle frame from the actual video (bottom-25% band,
# t=40s of the same clip validated earlier with EasyOCR/RapidOCR) - white
# outlined text over a busy, colorful background (flowers, statue).
real_frame = Image.open(
    r"C:\Users\kanga\AppData\Local\Temp\ytt_tesseract_bench_frames\frame_000001.jpg"
)
real_frame

# Known ground truth for this frame: '今天為了寄託定弘'

In [ ]:
from PIL import ImageOps

print("=== Tesseract (chi_tra) on a REAL subtitle frame, several configs ===")
for psm in (6, 7, 11, 12):
    t0 = time.monotonic()
    text = pytesseract.image_to_string(real_frame, lang="chi_tra", config=f"--psm {psm}").strip()
    print(f"psm={psm} raw ({(time.monotonic()-t0)*1000:.0f}ms): {text!r}")

gray = ImageOps.grayscale(real_frame)
bw = gray.point(lambda p: 255 if p > 180 else 0)
for psm in (6, 7, 11):
    t0 = time.monotonic()
    text = pytesseract.image_to_string(bw, lang="chi_tra", config=f"--psm {psm}").strip()
    print(f"psm={psm} thresholded ({(time.monotonic()-t0)*1000:.0f}ms): {text!r}")

# LOGGED RESULT - ground truth was '今天為了寄託定弘':
# psm=6  raw: '及 說\n7\nAA\n人 和 本'
# psm=7  raw: '0'
# psm=11 raw: '人\n\n,\n\n7 吧\n\nXX 補\n\n時\n\n~\n\n才\n\npe 人 3\n\n地 寺中\n\n人\n\n『「\n\n苞 >\n\n人\n\n才'
# psm=12 raw: (similar garbage)
# psm=6  thresholded: '全 和 有\n全'
# psm=7  thresholded: '了 3 生 卻 人 才 才'
# psm=11 thresholded: '全\n\n有\n\n人\n\nsa\n\n友\n\nVs\n\n2\n\n還\n\n了 蟬 伯 旬\n\n開放 和 詢'
#
# CONCLUSION: every config either finds nothing or hallucinates characters
# from the busy background (flowers/statue texture) instead of the actual
# white subtitle text - not a PSM tuning issue, Tesseract's classical
# (non-deep-learning) approach just can't separate subtitle text from a
# cluttered video background the way EasyOCR's CRAFT detector + CNN/RNN
# recognizer does. Per-call latency (~650-1000ms) IS much faster than
# EasyOCR, but the output is unusable on real content -> not viable here,
# regardless of speed.

## RapidOCR (ONNX, PP-OCRv4 "ch" model)

Run separately via `backend/tests/benchmark_ocr_engines.py` (isolated `.venv_bench`, since it needs different deps than EasyOCR). Logged results below.

In [ ]:
# Fixtures (from backend/tests/benchmark_ocr_engines.py, run in .venv_bench):
#   greeting.jpg: 922ms  text='你好，歡迎收看'   match=True
#   question.jpg: 469ms  text='這是什意思？'    match=False  (dropped 麼: 這是什麼意思 -> 這是什意思)
#   mixed.jpg:   1141ms  text='第3集-再見'      match=True
#   total: 2532ms for 3 images (844ms/image avg)
#
# Real subtitle frames, same clip validated with EasyOCR (via
# backend/tests/run_real_subtitle_check.py using RapidOCR):
#   EasyOCR:   '定弘這三年來 幾乎日日都思念 我們尊敬的恩師上y 我們尊敬的恩師上入
#               念念想著要如何報恩 今天為了寄詫定弘 對老入家永遠的懷念和哀思
#               啟講(無量壽經)'
#   RapidOCR:  '定弘这三 年来 襄乎日日都思念 襄乎日旧都思念 我們尊敬的恩師上人 Y
#               念念想著要如何报恩 街老人家水康的健念和哀思
#               野老人家水速的健念和哀思 啟講 《無量經》'
#
# CONCLUSION: real misreadings on real video content, not just a
# Simplified/Traditional style difference - 幾(correct)->襄(wrong), 對->街/野,
# 永遠->水康/水速, a whole line ("今天為了寄詫定弘") dropped entirely, 壽
# dropped from the title. Much faster (~3-6x on clean fixtures, no torch
# dependency) but not accurate enough on this real, cluttered-background
# subtitle content - the whole point of this feature is OCR overriding
# Whisper on disagreement, so garbled OCR text would override *correct*
# Whisper output. Reverted.
print("(see comment above - results transcribed from the actual run log)")

## PaddleOCR (chinese_cht - dedicated Traditional Chinese model)

Run separately via `backend/tests/benchmark_paddleocr.py` in an isolated `.venv_bench` (`pip install paddlepaddle paddleocr`), since it's a large, separate dependency stack. PaddleOCR 3.7.0 installed.

In [ ]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    lang="chinese_cht",
    use_textline_orientation=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
)
result = ocr.predict("fixtures/greeting.jpg")
print(result[0]["rec_texts"] if result else "NO RESULT")

# LOGGED RESULT - every model version crashes on actual inference (not an
# accuracy issue - it never produces output at all):
#
#   load time: 37.0s (first run, downloads PP-OCRv6_medium_det +
#               PP-OCRv6_medium_rec model weights - chinese_cht is only
#               available on PP-OCRv6 in this PaddleOCR version)
#
#   NotImplementedError: (Unimplemented)
#     ConvertPirAttribute2RuntimeAttribute not support
#     [pir::ArrayAttribute<pir::DoubleAttribute>]
#     (at ..\paddle\fluid\framework\new_executor\instruction\onednn\onednn_instruction.cc:118)
#
# Tried forcing older model versions (ocr_version='PP-OCRv3' / 'PP-OCRv5')
# to rule out a v6-specific bug - construction succeeds for all three, but
# inference (.predict()) crashes with the IDENTICAL oneDNN error on every
# version. This is a fundamental paddlepaddle CPU/oneDNN backend
# incompatibility in this environment (Windows CPU wheel), not something
# fixable via config - PaddleOCR could not be benchmarked for accuracy at
# all here, regardless of speed potential.
print("(see comment above - results transcribed from the actual run log)")

## Summary

| Engine | ~ms/frame (fixtures) | Real-frame result | Verdict |
|---|---|---|---|
| **EasyOCR** (current, `ch_tra`) | ~2700-3000 (with 3-way concurrency) | Correct, repeatedly validated | Keep - too slow for long videos, but works |
| RapidOCR (`ch`, ONNX) | ~500-1100 | Real misreadings, a dropped line, dropped characters | Not accurate enough |
| Tesseract (`chi_tra`) | ~650-1000 | Empty / hallucinated garbage from background clutter | Not viable at all on real frames |
| PaddleOCR (`chinese_cht`) | n/a | **Crashes** on every model version (oneDNN CPU bug) | Not runnable in this environment |

None of the three faster alternatives are usable as-is. EasyOCR remains the only engine that reads real subtitle frames (busy, colorful video backgrounds) correctly. The path to a 92-minute video finishing in reasonable time has to come from **doing less work**, not swapping engines:
- Skip near-duplicate consecutive frames (subtitle lines persist ~3-4s, we sample every 2s)
- Reduce sample rate
- Downscale crops before OCR

All of these keep EasyOCR's accuracy intact.